# LeetCode #194 — Transpose File: chuyển vị một file văn bản bằng `awk`

**Mức độ:** Medium · **Chủ đề:** Shell (Bash)

Bài **Shell**: nộp một lệnh Bash đọc `file.txt`. Notebook chạy lệnh Bash thật và viết bản Python tương đương.

Skeleton trên LeetCode:

```bash
# Read from the file file.txt and print its transposed content to stdout.
```

Mục tiêu của notebook:

1. Hiểu **chuyển vị** (transpose): hàng thành cột, cột thành hàng — giống ma trận.
2. `awk` xử lý **từng dòng** một: biến `NR` (số thứ tự dòng), `NF` (số cột), `$i` (cột thứ `i`), mảng kết hợp, khối `END`.
3. Ý tưởng: tích lũy chuỗi kết quả cho **mỗi cột** trong một mảng, in ra ở `END`.
4. Cách khác: lặp theo cột với `cut` + `paste`. Bản Python `zip(*rows)`.

# 1. Bài toán là gì?

Cho file văn bản `file.txt`, in nội dung **đã chuyển vị** của nó.

Giả thiết: mỗi dòng có **cùng số cột**, các cột cách nhau bởi **một** dấu cách `' '`.

---

## Ví dụ

```text
file.txt:            Kết quả:
name age             name alice ryan
alice 21             age 21 30
ryan 30
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Xử lý văn bản bằng Bash (pipe, tr, sort, uniq, awk)
2. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Xử lý văn bản bằng Bash (pipe, tr, sort, uniq, awk)

### Triết lý Unix: nhiều lệnh nhỏ nối bằng **ống** `|`

Mỗi lệnh đọc văn bản từ **đầu vào chuẩn**, ghi ra **đầu ra chuẩn**; dấu `|` nối đầu ra lệnh trước vào đầu vào lệnh sau.

```text
cat words.txt | tr -s ' ' '\n' | sort | uniq -c | sort -rn
  đọc file      mỗi từ 1 dòng    xếp   đếm liền   xếp theo số giảm dần
```

| Lệnh | Tác dụng |
|---|---|
| `cat f` | in nội dung file |
| `tr -s ' ' '\n'` | thay ký tự (`-s`: gộp các ký tự lặp liên tiếp) |
| `sort` / `sort -rn` / `sort -k2` | sắp xếp (theo chữ / số giảm dần / theo cột 2) |
| `uniq -c` | gộp các dòng **giống nhau liền kề**, đếm số lần |
| `head -n 5`, `tail -n +2` | lấy 5 dòng đầu, bỏ dòng đầu |
| `awk '{...}'` | ngôn ngữ nhỏ xử lý từng dòng theo **trường** |

Trong notebook ta gọi Bash qua `subprocess`; nếu máy không có `bash` thì phần demo được bỏ qua (bản Python vẫn chạy).

In [1]:
import shutil, subprocess, os, tempfile

HAS_BASH_DEMO = shutil.which("bash") is not None

def bash_demo(cmd, files):
    if not HAS_BASH_DEMO:
        print("(máy không có bash — bỏ qua)")
        return
    with tempfile.TemporaryDirectory() as d:
        for name, content in files.items():
            with open(os.path.join(d, name), "w") as fh:
                fh.write(content)
        r = subprocess.run(["bash", "-c", cmd], cwd=d, capture_output=True, text=True)
        print(r.stdout, end="")

text = "the day is sunny the the\nthe sunny is is\n"
bash_demo("cat words.txt | tr -s ' ' '\\n' | sort | uniq -c | sort -rn", {"words.txt": text})

      4 the
      3 is
      2 sunny
      1 day


### `uniq` chỉ gộp các dòng **liền kề** → phải `sort` trước

In [2]:
bash_demo("printf 'a\\nb\\na\\n' | uniq -c", {})
print("---- sau khi sort:")
bash_demo("printf 'a\\nb\\na\\n' | sort | uniq -c", {})

      1 a
      1 b
      1 a
---- sau khi sort:
      2 a
      1 b


### awk: xử lý theo trường

- `$1, $2, ...` là các **trường** (tách theo khoảng trắng), `$0` là cả dòng, `NF` là số trường, `NR` là số thứ tự dòng.
- Khối `BEGIN {...}` chạy trước dòng đầu, `END {...}` chạy sau dòng cuối.
- Mảng của awk là **mảng kết hợp** (giống `dict`): `count[$1]++`.

In [3]:
bash_demo("awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt | sort -k2 -rn",
          {"words.txt": text})
bash_demo("awk '{ print NR \": \" NF \" từ, từ cuối = \" $NF }' words.txt", {"words.txt": text})

the 4
is 3
sunny 2
day 1
1: 6 từ, từ cuối = the
2: 4 từ, từ cuối = is


### Tương đương bằng Python

In [4]:
from collections import Counter

cnt = Counter(text.split())
for w, c in sorted(cnt.items(), key=lambda kv: -kv[1]):
    print(w, c)

the 4
is 3
sunny 2
day 1


### Lỗi hay gặp

- `uniq -c` khi chưa `sort` → đếm sai.
- `sort` mặc định so sánh **chữ** (`"10" < "9"`) → dùng `-n` cho số.
- `tr ' ' '\n'` không có `-s` → nhiều khoảng trắng liên tiếp sinh **dòng rỗng**.
- Trích dẫn: lệnh awk đặt trong `'...'` để shell không thay `$1`.

## 2.2 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [5]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [6]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [7]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [8]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.3 Áp dụng vào bài này

Chuyển vị file = **chuyển vị ma trận** (hàng ↔ cột) với mỗi dòng là một hàng, mỗi từ là một ô. Trong `awk`, mỗi dòng có `NF` trường `$1..$NF`; ta nối trường thứ `i` vào chuỗi `s[i]` (mảng kết hợp), và in mọi `s[i]` trong khối `END`. Bản Python: `zip(*rows)`.

# 3. Nền tảng

In [9]:
import os, random, shutil, subprocess, tempfile

HAS_BASH = shutil.which("bash") is not None


def run_bash(cmd: str, files: dict) -> str:
    if not HAS_BASH:
        raise RuntimeError("máy này không có bash")
    with tempfile.TemporaryDirectory() as d:
        for name, content in files.items():
            with open(os.path.join(d, name), "w") as f:
                f.write(content)
        r = subprocess.run(["bash", "-c", cmd], cwd=d, capture_output=True, text=True)
        return r.stdout + (("[stderr] " + r.stderr) if r.stderr else "")


EXAMPLE = "name age\nalice 21\nryan 30\n"
print("có bash:", HAS_BASH)

có bash: True


## Chuyển vị trong Python

`zip(*rows)` ghép phần tử thứ `i` của mọi hàng thành một bộ — chính là cột `i`.

In [10]:
rows = [line.split(" ") for line in EXAMPLE.splitlines()]
print("hàng:", rows)
print("cột :", list(zip(*rows)))

def transpose_py(text: str) -> str:
    rows = [line.split(" ") for line in text.splitlines()]
    return "".join(" ".join(col) + "\n" for col in zip(*rows))

print(transpose_py(EXAMPLE))

hàng: [['name', 'age'], ['alice', '21'], ['ryan', '30']]
cột : [('name', 'alice', 'ryan'), ('age', '21', '30')]
name alice ryan
age 21 30



## `awk` hoạt động thế nào?

`awk 'chương trình' file` đọc file **từng dòng**; với mỗi dòng nó tách thành các cột theo khoảng trắng rồi chạy chương trình.

| Ký hiệu | Ý nghĩa |
|---|---|
| `$1, $2, …, $NF` | cột thứ 1, 2, …, cột cuối |
| `NF` | số cột của dòng hiện tại |
| `NR` | số thứ tự dòng hiện tại (bắt đầu từ 1) |
| `s[i]` | mảng kết hợp (giống `dict`), không cần khai báo |
| `BEGIN { … }` / `END { … }` | chạy trước dòng đầu / sau dòng cuối |
| `a " " b` | ghép chuỗi (đặt cạnh nhau) |

In [11]:
if HAS_BASH:
    print(run_bash("awk '{ print NR \": NF =\", NF, \"| cột 1 =\", $1, \"| cột cuối =\", $NF }' file.txt", {"file.txt": EXAMPLE}))

1: NF = 2 | cột 1 = name | cột cuối = age
2: NF = 2 | cột 1 = alice | cột cuối = 21
3: NF = 2 | cột 1 = ryan | cột cuối = 30



# 4. Ý tưởng chính

Duyệt từng dòng; với mỗi cột `i` của dòng, **nối** giá trị `$i` vào cuối chuỗi `s[i]` (chuỗi này sẽ là **dòng thứ `i`** của kết quả). Dòng đầu tiên thì khởi tạo `s[i] = $i`, các dòng sau thì `s[i] = s[i] " " $i`. Đọc xong file (khối `END`), in `s[1], s[2], …, s[NF]`.

```bash
awk '{
    for (i = 1; i <= NF; i++) {
        if (NR == 1) s[i] = $i
        else         s[i] = s[i] " " $i
    }
}
END {
    for (i = 1; i <= NF; i++) print s[i]
}' file.txt
```

Trong khối `END`, `NF` vẫn giữ số cột của **dòng cuối** (mọi dòng cùng số cột nên dùng được).

In [12]:
SOLUTION = """awk '{
    for (i = 1; i <= NF; i++) {
        if (NR == 1) s[i] = $i
        else         s[i] = s[i] " " $i
    }
}
END {
    for (i = 1; i <= NF; i++) print s[i]
}' file.txt"""
if HAS_BASH:
    print(run_bash(SOLUTION, {"file.txt": EXAMPLE}))

name alice ryan
age 21 30


# 5. Mô phỏng: mảng `s` sau mỗi dòng

In [13]:
def trace_awk(text):
    s = {}
    for nr, line in enumerate(text.splitlines(), 1):
        cols = line.split()
        for i, v in enumerate(cols, 1):
            s[i] = v if nr == 1 else s[i] + " " + v
        print(f"sau dòng {nr} ({line!r}): s = {s}")
    print("END → in:", [s[i] for i in range(1, len(cols) + 1)])

trace_awk(EXAMPLE)

sau dòng 1 ('name age'): s = {1: 'name', 2: 'age'}
sau dòng 2 ('alice 21'): s = {1: 'name alice', 2: 'age 21'}
sau dòng 3 ('ryan 30'): s = {1: 'name alice ryan', 2: 'age 21 30'}
END → in: ['name alice ryan', 'age 21 30']


# 6. Cách khác: lặp theo cột với `cut` và `paste`

- `cut -d' ' -fK file.txt` lấy cột `K` (mỗi giá trị một dòng).
- `paste -sd' '` ghép các dòng của đầu vào thành **một** dòng, ngăn bởi dấu cách.

```bash
ncol=$(head -1 file.txt | wc -w)
for i in $(seq 1 $ncol); do
    cut -d' ' -f$i file.txt | paste -sd' '
done
```

Dễ đọc nhưng đọc file `ncol` lần → chậm hơn `awk` (đọc một lần) khi có nhiều cột.

In [14]:
SOLUTION_CUT = """ncol=$(head -1 file.txt | wc -w)
for i in $(seq 1 $ncol); do
    cut -d' ' -f$i file.txt | paste -sd' '
done"""
if HAS_BASH:
    print(run_bash(SOLUTION_CUT, {"file.txt": EXAMPLE}))

name alice ryan
age 21 30



# 7. Phiên bản nộp LeetCode

```bash
awk '{ for (i = 1; i <= NF; i++) { if (NR == 1) s[i] = $i; else s[i] = s[i] " " $i } } END { for (i = 1; i <= NF; i++) print s[i] }' file.txt
```

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS")


def random_file():
    r, c = random.randint(1, 5), random.randint(1, 5)
    return "".join(" ".join(random.choice(["a", "bb", "7", "x9"]) for _ in range(c)) + "\n" for _ in range(r))


check("Python, ví dụ", transpose_py(EXAMPLE), "name alice ryan\nage 21 30\n")
if HAS_BASH:
    check("awk, ví dụ", run_bash(SOLUTION, {"file.txt": EXAMPLE}), "name alice ryan\nage 21 30\n")
    check("cut+paste, ví dụ", run_bash(SOLUTION_CUT, {"file.txt": EXAMPLE}), "name alice ryan\nage 21 30\n")
    check("một dòng", run_bash(SOLUTION, {"file.txt": "a b c\n"}), "a\nb\nc\n")
    check("một cột", run_bash(SOLUTION, {"file.txt": "a\nb\nc\n"}), "a b c\n")
    random.seed(194)
    for _ in range(100):
        f = random_file()
        exp = transpose_py(f)
        assert run_bash(SOLUTION, {"file.txt": f}) == exp == run_bash(SOLUTION_CUT, {"file.txt": f}), f
    print("100 file ngẫu nhiên: hai lệnh Bash khớp bản Python")
else:
    print("(bỏ qua phần Bash vì không có bash)")

Python, ví dụ        PASS
awk, ví dụ           PASS


cut+paste, ví dụ     PASS


một dòng             PASS
một cột              PASS


100 file ngẫu nhiên: hai lệnh Bash khớp bản Python


# 8. Vì sao đúng? Độ phức tạp?

Sau khi đọc dòng `r`, `s[i]` là chuỗi `a[1][i] a[2][i] … a[r][i]` (các phần tử cột `i` của `r` dòng đầu, ngăn bởi dấu cách) — chứng minh bằng quy nạp theo `r`. Khi hết file, `s[i]` đúng là dòng `i` của ma trận chuyển vị.

```text
Thời gian: O(R · C) với R dòng, C cột (đọc file một lần)
Bộ nhớ:    O(R · C) — phải giữ toàn bộ nội dung trước khi in dòng đầu tiên của kết quả
```

Không thể in kết quả "vừa đọc vừa in": dòng đầu của kết quả cần phần tử đầu của **mọi** dòng — phải đọc hết file trước.

# 9. Những lỗi dễ mắc

## Lỗi 1: Luôn nối `" " $i`, kể cả dòng đầu

In [16]:
WRONG_SPACE = "awk '{ for (i = 1; i <= NF; i++) s[i] = s[i] \" \" $i } END { for (i = 1; i <= NF; i++) print s[i] }' file.txt"
if HAS_BASH:
    print(repr(run_bash(WRONG_SPACE, {"file.txt": EXAMPLE})))

' name alice ryan\n age 21 30\n'


Mỗi dòng kết quả có thêm **một dấu cách ở đầu** → sai định dạng. Cần trường hợp riêng cho `NR == 1` (hoặc in bằng `substr(s[i], 2)` để bỏ ký tự đầu).

## Lỗi 2: In ngay trong lúc đọc

In [17]:
WRONG_PRINT = "awk '{ for (i = 1; i <= NF; i++) print $i }' file.txt"
if HAS_BASH:
    print(repr(run_bash(WRONG_PRINT, {"file.txt": EXAMPLE})))

'name\nage\nalice\n21\nryan\n30\n'


Chỉ là "mỗi phần tử một dòng", không phải chuyển vị. Phải **tích lũy** trong mảng và in ở `END`.

## Lỗi 3: Dùng `for (i in s)` để in

`for (k in mảng)` của `awk` duyệt khóa **không theo thứ tự** đảm bảo → các dòng kết quả có thể bị đảo. Dùng vòng `for (i = 1; i <= NF; i++)`.

In [18]:
if HAS_BASH:
    many = " ".join(str(k) for k in range(1, 13)) + "\n" + " ".join(str(k * 10) for k in range(1, 13)) + "\n"
    out_in = run_bash("awk '{ for (i = 1; i <= NF; i++) s[i] = (NR == 1 ? $i : s[i] \" \" $i) } END { for (i in s) print s[i] }' file.txt", {"file.txt": many})
    print("for (i in s):", out_in.split("\n")[:4], "...")
    print("đúng thứ tự?", out_in == transpose_py(many))

for (i in s): ['3 30', '10 100', '8 80', '6 60'] ...
đúng thứ tự? False


Kết quả của `for (i in s)` phụ thuộc vào cài đặt `awk` (gawk, mawk, …): có máy ra đúng thứ tự, có máy không. Đừng dựa vào điều không được đảm bảo.

# 10. Mẹo nhận diện pattern

- **Cần toàn bộ dữ liệu trước khi xuất** → tích lũy trong mảng, xuất ở `END`.
- **Theo cột trong Shell** → `awk '{… $i …}'`, `cut -d -f`, `paste`.
- **Chuyển vị trong Python** → `zip(*rows)`.

Bài liên quan: **#867 Transpose Matrix** (Easy), **#48 Rotate Image**, **#192 Word Frequency**, **#195 Tenth Line** (Easy).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chuyển vị file `"1 2 3\n4 5 6\n"` và ghi lại `s` sau mỗi dòng.

## Bài 2 — Bản Python

In [19]:
def exercise_2_transpose(text: str) -> str:
    # TODO: trả về đúng chuỗi lệnh Bash in ra (mỗi dòng kết thúc bằng "\n")
    pass


# assert exercise_2_transpose("name age\nalice 21\nryan 30\n") == "name alice ryan\nage 21 30\n"
# assert exercise_2_transpose("a\nb\n") == "a b\n"

## Bài 3 — Bash: in **cột thứ 2** của file thành một dòng

In [20]:
def exercise_3_second_column_cmd() -> str:
    # TODO
    pass


# assert (not HAS_BASH) or run_bash(exercise_3_second_column_cmd(), {"file.txt": "name age\nalice 21\nryan 30\n"}) == "age 21 30\n"

## Bài 4 — Biến thể Python: các dòng **có số cột khác nhau** (thiếu thì bỏ qua)

`"a b c\nd\ne f\n"` → cột 1: `a d e`, cột 2: `b f`, cột 3: `c`.

In [21]:
def exercise_4_transpose_ragged(text: str) -> str:
    # TODO
    pass


# assert exercise_4_transpose_ragged("a b c\nd\ne f\n") == "a d e\nb f\nc\n"
# assert exercise_4_transpose_ragged("x\n") == "x\n"

## Bài 5 — Mini challenge: Bash cho bài 4

Gợi ý: giữ `max` = số cột lớn nhất đã gặp; khởi tạo `s[i]` khi nó **chưa có** (không dựa vào `NR == 1`).

In [22]:
def exercise_5_ragged_cmd() -> str:
    # TODO
    pass


# assert (not HAS_BASH) or run_bash(exercise_5_ragged_cmd(), {"file.txt": "a b c\nd\ne f\n"}) == "a d e\nb f\nc\n"

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_awk("1 2 3\n4 5 6\n")` sau khi tự làm.

**Bài 3**
- Mức 1: `cut -d' ' -f2 file.txt | paste -sd' '`, hoặc `awk '{ printf ... }'`.

**Bài 4**
- Mức 1: `itertools.zip_longest(*rows)` rồi bỏ các giá trị `None`.

**Bài 5**
- Mức 1: `if (NF > max) max = NF`; `if (i in s) s[i] = s[i] " " $i; else s[i] = $i`; ở `END` lặp `i = 1..max`.
- Mức 2: **bẫy**: viết gọn `s[i] = (i in s) ? … : $i` sẽ sai — `awk` tạo phần tử `s[i]` (rỗng) khi nhìn thấy vế trái **trước** khi kiểm tra `i in s`, nên điều kiện luôn đúng và kết quả có dấu cách thừa ở đầu.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
trace_awk("1 2 3\n4 5 6\n")

sau dòng 1 ('1 2 3'): s = {1: '1', 2: '2', 3: '3'}
sau dòng 2 ('4 5 6'): s = {1: '1 4', 2: '2 5', 3: '3 6'}
END → in: ['1 4', '2 5', '3 6']


## Đáp án Bài 2

In [24]:
def answer_2_transpose(text: str) -> str:
    return transpose_py(text)

assert answer_2_transpose("name age\nalice 21\nryan 30\n") == "name alice ryan\nage 21 30\n"
assert answer_2_transpose("a\nb\n") == "a b\n"
print("Bài 2 PASS (giống transpose_py ở mục 3)")

Bài 2 PASS (giống transpose_py ở mục 3)


## Đáp án Bài 3

In [25]:
def answer_3_second_column_cmd() -> str:
    return "cut -d' ' -f2 file.txt | paste -sd' '"

assert (not HAS_BASH) or run_bash(answer_3_second_column_cmd(), {"file.txt": "name age\nalice 21\nryan 30\n"}) == "age 21 30\n"
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
from itertools import zip_longest

def answer_4_transpose_ragged(text: str) -> str:
    rows = [line.split() for line in text.splitlines()]
    return "".join(" ".join(v for v in col if v is not None) + "\n" for col in zip_longest(*rows))

assert answer_4_transpose_ragged("a b c\nd\ne f\n") == "a d e\nb f\nc\n"
assert answer_4_transpose_ragged("x\n") == "x\n"
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [27]:
def answer_5_ragged_cmd() -> str:
    return ("awk '{ if (NF > max) max = NF; for (i = 1; i <= NF; i++) { if (i in s) s[i] = s[i] \" \" $i; else s[i] = $i } } "
            "END { for (i = 1; i <= max; i++) print s[i] }' file.txt")

assert (not HAS_BASH) or run_bash(answer_5_ragged_cmd(), {"file.txt": "a b c\nd\ne f\n"}) == "a d e\nb f\nc\n"
if HAS_BASH:
    for _ in range(100):
        f = "".join(" ".join(random.choice("abc") for _ in range(random.randint(1, 4))) + "\n" for _ in range(random.randint(1, 5)))
        assert run_bash(answer_5_ragged_cmd(), {"file.txt": f}) == answer_4_transpose_ragged(f), f
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> `awk` đọc từng dòng; với mỗi cột `i`, nối `$i` vào `s[i]` (dòng đầu thì gán, dòng sau thì thêm `" " $i`); ở `END` in `s[1..NF]` theo thứ tự. Python: `zip(*rows)`. Cách `cut -f$i | paste -sd' '` dễ đọc nhưng đọc file nhiều lần.

---

## Checklist trước khi nộp

- [ ] Dòng đầu gán, các dòng sau thêm dấu cách (không có dấu cách thừa ở đầu)?
- [ ] In ở khối `END`?
- [ ] In theo thứ tự `i = 1..NF` (không `for (i in s)`)?